# 0. Bibliotecas

Nesta célula importamos todas as bibliotecas necessárias para o projeto:

- **pandas** — manipulação e análise de dados em tabelas (DataFrames)
- **numpy** — operações matemáticas e arrays numéricos
- **re** — expressões regulares (para limpeza/parsing de texto)
- **sklearn.preprocessing** — ferramentas de pré-processamento:
  - `LabelEncoder` — converte categorias (texto) em números inteiros

> Nota: este notebook faz apenas preparação/limpeza; o treino dos modelos é feito nos notebooks de ML (Decision Tree / Random Forest / SVM / XGBoost).


In [1]:
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Importação dos dados

Os dados encontram-se no ficheiro `DATASET_ALEX.csv` (separador `;`).

O que este bloco faz:
1. Lê o CSV para um DataFrame `df_raw`
2. Mostra a dimensão do dataset e um preview inicial

> Se o separador estiver errado, o `shape` e as colunas serão um bom indicador (muitas colunas coladas numa só).


In [2]:
# -------- 1) Ler dataset (export com separador ;) --------
RAW_CANDIDATES = [
    Path('DATASET_ALEX.csv'),
    Path('Projeto') / 'DATASET_ALEX.csv',
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_PATH is None:
    tried = ', '.join(str(p) for p in RAW_CANDIDATES)
    raise FileNotFoundError(f'Não encontrei DATASET_ALEX.csv. Tentei: {tried}')

BASE_DIR = RAW_PATH.parent.resolve()

df_raw = pd.read_csv(RAW_PATH, sep=';')
print('BASE_DIR:', BASE_DIR)
print('raw shape:', df_raw.shape)
df_raw.head(3)

BASE_DIR: C:\Users\alexm\OneDrive\Documentos\GitHub\Automatic_Classification_BIM\Projeto
raw shape: (80646, 19)


,[I] Family and Type,[I] Type,[I] Workset,[I] Category,[I] IfcGUID,[I] Level,[I] Mark,[T] Type IfcGUID,[T] Workset,[T] Family Name,[I] Construction Block,[T] Classification Number,[T] Type Name,[T] Category,[I] Type Id,[I] Volume,Target (EF_CODE),Target (Ss_CODE),Target (Pr_CODE)
0,TPF-ValveBox.Xylem-Standard-GEOM: Standard,Standard,DRG-Exterior,Generic Models,082Y9U8n57RxOQ13jCCxiV,Ground Floor,NaN,082Y9U8n57RxOQ13jCCxXM,Family : Generic Models : TPF-ValveBox.Xylem-...,TPF-ValveBox.Xylem-Standard-GEOM,NaN,NaN,Standard,Generic Models,1731873.0,0.04 m³,0,0,0
1,TPF-Wastewater-Contaminated-Station.Flygt-Auto...,Standard,DRG-Exterior,Generic Models,082Y9U8n57RxOQ13jCCxWl,Ground Floor,NaN,082Y9U8n57RxOQ13jCCx7K,Family : Generic Models : TPF-Wastewater-Cont...,TPF-Wastewater-Contaminated-Station.Flygt-Auto...,NaN,NaN,Standard,Generic Models,1729699.0,0.76 m³,0,0,0
2,TPF-InspectionBox-Blind-RefLevels: 0.60 x 0.60,0.60 x 0.60,DRG,Mechanical Equipment,1uIinGoWPC5RVu5GeQ3hP0,Ground Floor,CVO3C01.3,3BOmqxrYf5hxpVbPIqK0b6,Family : Mechanical Equipment : TPF-Inspectio...,TPF-InspectionBox-Blind-RefLevels,O3,NaN,0.60 x 0.60,Mechanical Equipment,1744703.0,0.85 m³,0,0,0


# 2. Tratamento das Variáveis

> **Requisito**: não descartar nenhuma linha do `DATASET_ALEX.csv`.

> Nesta preparação nós **não removemos linhas** nem “apagamos” classes. Em vez disso, criamos *flags* para que os notebooks de treino saibam quais linhas são treináveis (supervisionado) e quais devem ser excluídas **apenas do subset de treino**.

**Pipeline aplicado (versão ALEX):**
1. Converter `"[I] Volume"` para uma variável numérica `Volume_m3`
2. Construir `target_valid` (target existe, não vazio e diferente de `0`)
3. Marcar `is_rare_class` (classes com poucos exemplos, útil para tratar SMOTE)
4. Construir as features `X` (numéricas + one-hot de algumas colunas categóricas do export)
5. Aplicar `LabelEncoder` ao target **apenas nas linhas com `target_valid=True`**
6. Criar `y_enc` com o mesmo tamanho do dataset, usando `-1` para linhas sem target válido


## 2.1 Volume — conversão para float (m³)

No export, o volume pode vir como texto (ex.: `"0.04 m³"`, `"1,23 m³"`) ou já numérico.

Para uniformizar:
- Removemos a unidade (`m³` / `m3`)
- Normalizamos vírgulas para ponto decimal
- Mantemos apenas caracteres numéricos relevantes
- Convertimos para `float` com `errors='coerce'` (valores inválidos viram `NaN`)


In [11]:
# -------- 2) Parsing de Volume (m³) para float --------
VOLUME_COL = '[I] Volume'

def parse_volume_m3(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    # exemplos: '0.04 m³', '1,23 m³', '0.04'
    s = s.replace('m³', '').replace('m3', '')
    s = s.replace(' ', '')
    s = s.replace(',', '.')
    # manter apenas números/sinal/ponto/expoente
    s = re.sub(r'[^0-9eE+.-]', '', s)
    return pd.to_numeric(s, errors='coerce')

df = df_raw.copy()
df['Volume_m3'] = df[VOLUME_COL].apply(parse_volume_m3)
print('Volume_m3 NaN %:', df['Volume_m3'].isna().mean() * 100)

Volume_m3 NaN %: 17.04486273342757


## 2.2 Target (Ss_CODE) — Variável Target (o que queremos prever)

O target no `DATASET_ALEX.csv` é `Target (Ss_CODE)` e representa a classificação **Ss** do elemento.

**Regras (versão ALEX, sem descartar linhas):**
1. Consideramos *válido* quando não é nulo, não é string vazia e não é `0`
2. Criamos a flag `target_valid` para indicar se cada linha é treinável
3. Calculamos classes raras (contagem apenas nas linhas válidas) e marcamos `is_rare_class`

> Nota: SMOTE e `train_test_split(..., stratify=...)` podem falhar em classes com muito poucas amostras. Por isso, os **notebooks de treino** criam um subset treinável excluindo (só para treino) targets inválidos e classes com <2 amostras.

In [12]:
# -------- 3) Target (Ss_CODE): limpeza + flags (sem descartar linhas) --------
TARGET_COL = 'Target (Ss_CODE)'
MIN_SAMPLES_PER_CLASS = 6  # usado para marcar classes raras (SMOTE pode falhar nelas)

y_raw = df[TARGET_COL].astype('string').str.strip()

# target válido = tem valor e não é '0' (tipicamente "sem classificação")
target_valid = y_raw.notna() & (y_raw != '') & (y_raw != '0')

# classes raras (contagem feita apenas nas linhas com target válido)
freq_valid = y_raw[target_valid].value_counts()
rare_classes = freq_valid[freq_valid < MIN_SAMPLES_PER_CLASS].index

is_rare_class = y_raw.isin(rare_classes)
df['target_valid'] = target_valid.astype(int)
df['is_rare_class'] = is_rare_class.astype(int)

print('total rows:', len(df))
print('target_valid rows:', int(target_valid.sum()))
print('n_classes_valid:', int(y_raw[target_valid].nunique()))
print('rare classes (valid target):', int(len(rare_classes)))
print('rows in rare classes:', int((target_valid & is_rare_class).sum()))

# preview das classes mais frequentes (apenas válidas)
freq_valid.head(15)

total rows: 80646
target_valid rows: 27888
n_classes_valid: 41
rare classes (valid target): 7
rows in rare classes: 16


Target (Ss_CODE)
Ss_65_40_33       6247
Ss_25_45          5232
Ss_32_46_65_55    3295
Ss_25_25_45_33    2567
Ss_25_25_45_35    2073
Ss_25_45_88       1503
Ss_25_50_45_35     697
Ss_20_30_75_65     639
Ss_25_45_70        629
Ss_25_13_50        589
Ss_30_40_30        558
Ss_25_45_72_28     550
Ss_32_46_65_58     527
Ss_25_10_20        386
Ss_60_60_70_94     334
Name: count, dtype: Int64

## 2.3 Features (X)

Construímos `X` a partir de:
- **Numéricas**: `Volume_m3`
- **Categóricas (one-hot)**: colunas de contexto do export (ex.: workset/categoria/nível), quando existirem.

> Para manter o notebook robusto, só usamos colunas que existam no CSV e criamos `dummy_na=True` para capturar valores em falta.


In [13]:
# -------- 4) Features: numéricas + one-hot de categóricas com baixa cardinalidade --------
ID_COL = '[I] IfcGUID'

categorical_cols = [
    '[I] Workset',
    '[I] Category',
    '[I] Level',
    '[I] Construction Block',
    '[T] Classification Number',
    '[T] Category',
]
categorical_cols = [c for c in categorical_cols if c in df.columns]

X_cat = pd.get_dummies(
    df[categorical_cols].astype('string'),
    prefix=[c.replace('[', '').replace(']', '') for c in categorical_cols],
    dummy_na=True,
 )
X_num = df[['Volume_m3']].copy()

X = pd.concat([X_num, X_cat], axis=1)
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(0)
print('X shape:', X.shape)

X shape: (80646, 172)


## 2.4 Label encoding do target

Os algoritmos de ML trabalham com targets numéricos.

O `LabelEncoder` converte cada classe em um inteiro sequencial, por exemplo:

```
"Ss-20" → 0
"Ss-30" → 1
"Ss-40" → 2
```

O mapeamento `label → encoded` deve ser guardado (ou recriado de forma consistente) para:
- interpretar métricas por classe
- converter previsões numéricas de volta para o código Ss


In [14]:
# -------- 5) Label encoding do target (sem descartar linhas) --------
le = LabelEncoder()

# Fit apenas em targets válidos
y_raw = df[TARGET_COL].astype('string').str.strip()
target_valid = df['target_valid'].astype(bool)
le.fit(y_raw[target_valid].astype(str))

# y_enc: -1 para linhas sem target válido
y_enc = np.full(len(df), -1, dtype=int)
y_enc[target_valid.to_numpy()] = le.transform(y_raw[target_valid].astype(str))

id_col = '[I] IfcGUID'
if id_col in df.columns:
    base = df[[id_col]].rename(columns={id_col: 'IfcGUID'}).reset_index(drop=True)
else:
    base = pd.DataFrame(index=range(len(df)))

preview = pd.concat(
    [
        base,
        y_raw.reset_index(drop=True).rename('Target_raw'),
        pd.Series(y_enc, name='Target_encoded'),
        df[['target_valid', 'is_rare_class']].reset_index(drop=True),
        X.reset_index(drop=True),
    ],
    axis=1,
 )

label_mapping = pd.DataFrame({'label': le.classes_, 'encoded': np.arange(len(le.classes_))})

print('preview shape:', preview.shape)
print('n_classes (valid):', len(le.classes_))
print('n_invalid_targets:', int((y_enc == -1).sum()))
label_mapping.head(10)

preview shape: (80646, 177)
n_classes (valid): 41
n_invalid_targets: 52758


,label,encoded
0,Ss_15_30_17_16,0
1,Ss_20_30_75_45,1
2,Ss_20_30_75_65,2
3,Ss_25_10_20,3
4,Ss_25_10_35_95,4
5,Ss_25_11_16,5
6,Ss_25_12_15_15,6
7,Ss_25_12_85_63,7
8,Ss_25_13_50,8
9,Ss_25_25_45_33,9


# 3. Export dos artefactos (para treino)

Nesta secção gravamos os dados já preparados para serem consumidos pelos notebooks de treino (Decision Tree / Random Forest / SVM / XGBoost), evitando repetir a preparação em cada notebook.

O ficheiro gerado fica em `Projeto/artifacts/prepared/alex_prepared.joblib` e contém apenas dados **portáteis** (para evitar problemas entre versões do pandas):
- `X_values` (`numpy.ndarray`, float32)
- `feature_cols` (lista com os nomes das colunas)
- `y_enc` (`numpy.ndarray`, int; `-1` = target inválido)
- `y_raw` (lista de strings, útil para inspeção)
- `target_valid` (`numpy.ndarray`, bool)
- `is_rare_class` (`numpy.ndarray`, bool)
- `label_classes` (classes do `LabelEncoder`, para recriar o encoder nos notebooks de treino)
- metadados (`target_col`, `raw_path`, `n_rows`)


In [17]:
# -------- 6) Export: artefactos para treino (compatível entre ambientes) --------
# Nota: evitamos guardar objetos pandas dentro do joblib (pode falhar entre versões).
# Guardamos apenas numpy arrays + listas + metadados simples.
PREP_DIR = BASE_DIR / 'artifacts' / 'prepared'
PREP_DIR.mkdir(parents=True, exist_ok=True)

y_raw = df[TARGET_COL].astype('string').str.strip().fillna('')
target_valid = df['target_valid'].astype(bool)
is_rare_class = df['is_rare_class'].astype(bool)

bundle = {
    'X_values': X.to_numpy(dtype=np.float32, copy=True),
    'feature_cols': list(X.columns),
    'y_enc': np.asarray(y_enc, dtype=int),
    'y_raw': y_raw.astype(str).tolist(),
    'target_valid': target_valid.to_numpy(dtype=bool, copy=True),
    'is_rare_class': is_rare_class.to_numpy(dtype=bool, copy=True),
    'label_classes': list(getattr(le, 'classes_', [])),
    'target_col': TARGET_COL,
    'raw_path': str(RAW_PATH),
    'n_rows': int(len(df)),
}

out_path = PREP_DIR / 'alex_prepared.joblib'
joblib.dump(bundle, out_path)

# Útil para inspeção manual
label_mapping.to_csv(PREP_DIR / 'label_mapping.csv', index=False)

print('Saved:', out_path)
print('rows:', bundle['n_rows'])
print('X_values shape:', bundle['X_values'].shape)
print('n_classes (valid):', len(bundle['label_classes']))
print('n_invalid_targets:', int((bundle['y_enc'] == -1).sum()))

Saved: C:\Users\alexm\OneDrive\Documentos\GitHub\Automatic_Classification_BIM\Projeto\artifacts\prepared\alex_prepared.joblib
rows: 80646
X_values shape: (80646, 172)
n_classes (valid): 41
n_invalid_targets: 52758


# 4. Export no formato `Tese_Final` (data4_x)

O pipeline da pasta `Tese_Final/` espera um CSV no formato `data4_1.csv`/`data4_2.csv` (com colunas geométricas + one-hot de categorias + target codificado).

O `DATASET_ALEX.csv` não tem a maioria das features geométricas do `Tese_Final`, por isso aqui exportamos um CSV *compatível* (mesmas colunas) preenchendo as features em falta com `0` (para o `SimpleImputer(median)` não falhar).

O target usado é `Target (Ss_CODE)` mapeado para:
- `SECClasS_Code_Ss_1` (label original)
- `SECClasS_Code_Ss_1_encode` (inteiro via `LabelEncoder`)

In [ ]:
# -------- 7) Export compatível com Tese_Final/data4_1.csv (sem descartar linhas) --------
from pathlib import Path

# localizar Tese_Final/ (a partir do Projeto/)
FINAL_CANDIDATES = [
    BASE_DIR.parent / 'Tese_Final',
    Path.cwd() / 'Tese_Final',
    Path.cwd().parent / 'Tese_Final',
]
TESE_FINAL_DIR = next((p for p in FINAL_CANDIDATES if p.exists()), None)
if TESE_FINAL_DIR is None:
    raise FileNotFoundError('Não encontrei a pasta Tese_Final/. Confirma a estrutura do workspace.')

out_csv = TESE_FINAL_DIR / 'data_alex_1.csv'

# colunas esperadas no formato data4_1.csv
FINAL_COLS = [
    'ElementID',
    'Volume', 'Area', 'Length', 'Height', 'Thickness/Width',
    'Total_Surface_Area', 'Total_Edge_Length',
    'Base_offset', 'Top_offset',
    'Number_of_Faces',
    'Bounding_Box_Width', 'Bounding_Box_Height', 'Bounding_Box_Depth',
    'Centroid_X', 'Centroid_Y', 'Centroid_Z',
    'Orientation_Angle', 'Curvature',
    'source', 'is_rare_class',
    'SECClasS_Code_Ss_1', 'SECClasS_Code_Ss_1_encode',
    'Category_Curtain Panels', 'Category_Doors', 'Category_Floors', 'Category_Roofs', 'Category_Stairs',
    'Category_Structural Columns', 'Category_Structural Foundations', 'Category_Structural Framing',
    'Category_Walls', 'Category_Windows',
    'load_bearing_status_binary',
 ]

df_final = pd.DataFrame(index=df.index)

# IDs / origem
if '[I] IfcGUID' in df.columns:
    df_final['ElementID'] = df['[I] IfcGUID'].astype(str).values
else:
    df_final['ElementID'] = np.arange(len(df), dtype=int)

df_final['source'] = 'DATASET_ALEX'
df_final['is_rare_class'] = df['is_rare_class'].astype(int).values

# target (mantém todas as linhas; -1 indica alvo inválido)
y_raw = df[TARGET_COL].astype('string').str.strip().fillna('')
df_final['SECClasS_Code_Ss_1'] = y_raw.astype(str).values
df_final['SECClasS_Code_Ss_1_encode'] = np.asarray(y_enc, dtype=int)

# features numéricas (as que não existem no ALEX ficam a 0)
numeric_defaults = [
    'Area', 'Length', 'Height', 'Thickness/Width',
    'Total_Surface_Area', 'Total_Edge_Length',
    'Base_offset', 'Top_offset',
    'Number_of_Faces',
    'Bounding_Box_Width', 'Bounding_Box_Height', 'Bounding_Box_Depth',
    'Centroid_X', 'Centroid_Y', 'Centroid_Z',
    'Orientation_Angle', 'Curvature',
 ]
for col in numeric_defaults:
    df_final[col] = 0.0

df_final['Volume'] = df['Volume_m3'].fillna(0).astype(float).values

# one-hot de categorias (a partir de [I] Category)
cat_series = (
    df['[I] Category'].astype('string')
    if '[I] Category' in df.columns
    else pd.Series([''] * len(df), index=df.index, dtype='string')
).fillna('')

def cat_flag(token: str) -> np.ndarray:
    return cat_series.str.contains(token, case=False, regex=False).astype(int).values

df_final['Category_Curtain Panels'] = cat_flag('Curtain Panels')
df_final['Category_Doors'] = cat_flag('Doors')
df_final['Category_Floors'] = cat_flag('Floors')
df_final['Category_Roofs'] = cat_flag('Roofs')
df_final['Category_Stairs'] = cat_flag('Stairs')
df_final['Category_Structural Columns'] = cat_flag('Structural Columns')
df_final['Category_Structural Foundations'] = cat_flag('Structural Foundations')
df_final['Category_Structural Framing'] = cat_flag('Structural Framing')
df_final['Category_Walls'] = cat_flag('Walls')
df_final['Category_Windows'] = cat_flag('Windows')

# não existe no ALEX (por agora)
df_final['load_bearing_status_binary'] = 0

# garantir ordem e existência de todas as colunas
for col in FINAL_COLS:
    if col not in df_final.columns:
        df_final[col] = 0

df_final = df_final[FINAL_COLS]
df_final.to_csv(out_csv, index=False)

print('Saved (Tese_Final format):', out_csv)
print('shape:', df_final.shape)
print('n_invalid_targets:', int((df_final['SECClasS_Code_Ss_1_encode'] == -1).sum()))
df_final.head(3)

Saved (Tese_Final format): C:\Users\alexm\OneDrive\Documentos\GitHub\Automatic_Classification_BIM\Tese_Final\data_alex_1.csv
shape: (80646, 34)
n_invalid_targets: 52758


,ElementID,Volume,Area,Length,Height,Thickness/Width,Total_Surface_Area,Total_Edge_Length,Base_offset,Top_offset,...,Category_Doors,Category_Floors,Category_Roofs,Category_Stairs,Category_Structural Columns,Category_Structural Foundations,Category_Structural Framing,Category_Walls,Category_Windows,load_bearing_status_binary
0,082Y9U8n57RxOQ13jCCxiV,0.04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,082Y9U8n57RxOQ13jCCxWl,0.76,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,1uIinGoWPC5RVu5GeQ3hP0,0.85,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
